## Catalog and Schema

In [0]:
# Configuration
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("bronze_schema", "bronze", ["bronze", "gabrielajaniszews786_bronze"], "Bronze schema")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

BRONZE_SENSOR = f"{CATALOG}.{BRONZE_SCHEMA}.sensor_data"

In [0]:
# Creating Silver Schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

In [0]:
# Checking the Bronze layer contents

display(spark.sql(f"DESCRIBE {BRONZE_SENSOR}"))
display(spark.sql(f"SELECT * FROM {BRONZE_SENSOR} LIMIT 20"))
display(spark.sql(f"SELECT count(*) AS rows, COUNT(DISTINCT timestamp_utc, country) AS unique FROM {BRONZE_SENSOR}"))

## Creating a Silver Table

In [0]:
# Create the silver fact table with an explicit, enforced schema.
# NOT NULL on key columns is enforced by Delta (real schema enforcement).
# PRIMARY KEY is informational only in Databricks - dedup is done by MERGE, not by this constraint.
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.sensor_data")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.sensor_data (
  event_id              STRING      NOT NULL,
  timestamp_utc         TIMESTAMP   NOT NULL,
  site_name             STRING,
  country               STRING,
  bidding_zone          STRING,
  consumption_kwh       DECIMAL(10,4),
  pue                   DECIMAL(4,3),
  enqueued_ts           TIMESTAMP,
  ingestion_ts          TIMESTAMP,
  site_id               STRING     NOT NULL,
  reading_interval_s    INT,
  silver_processed_ts   TIMESTAMP,
  CONSTRAINT pk_sensor PRIMARY KEY (event_id)
  -- FK added AFTER dim_datacenter exists:
  -- , CONSTRAINT fk_site FOREIGN KEY (site_id) REFERENCES {CATALOG}.{SILVER_SCHEMA}.dim_datacenter
)
USING DELTA
""")


## Cleaning the Sensor Data

In [0]:
from pyspark.sql import functions as F, Window

# Reading the bronze source
bronze = spark.table(BRONZE_SENSOR)

# Deduplication rule: within each business key kepp the most recently ingested row
key_window = Window.partitionBy("event_id").orderBy(F.col("ingestion_ts").desc())

clean = (bronze
         # --- data quality filters that should not reach silver ---
         .filter(F.col("event_id").isNotNull()) # primary key must exist
         .filter(F.col("bidding_zone").isNotNull()) # FK must exist
         .filter(F.col("timestamp_utc").isNotNull()) # timestamp is required
         .withColumn("timestamp_utc", F.col("timestamp_utc").cast("timestamp")) # string > timestamp conversion
         # --- typing to match the silver schema established above ---
         .withColumn("consumption_kwh", F.col("consumption_kwh").cast("decimal(10,4)"))
         .withColumn("pue", F.col("pue").cast("decimal(4,3)"))
         .withColumn("reading_interval_s", F.col("reading_interval_s").cast("int"))
         # --- keep only the latest row per key ---
         .withColumn("row_num", F.row_number().over(key_window))
         .filter(F.col("row_num") == 1)
         # --- silver columns in the same order ---
         .selectExpr(
             "event_id",
             "timestamp_utc",
             "site_name",
             "country",
             "bidding_zone",
             "consumption_kwh",
             "pue",
             "enqueued_ts",
             "ingestion_ts",
             "site_id",
             "reading_interval_s",
             "current_timestamp() AS silver_processed_ts"        # when silver processed this row
    ))

clean.createOrReplaceTempView("silver_updates")

print("Rows after cleaning and deduplication:", clean.count())

## Upserting into Silver

In [0]:
# First run: nothing MATCHED -> INSERT.
# Re-run: everything is MATCHED -> UPDATE (idempotent, no duplicates).

spark.sql(f"""
    MERGE INTO {CATALOG}.{SILVER_SCHEMA}.sensor_data AS t
    USING silver_updates AS s
      ON t.event_id = s.event_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
# Verification
display(spark.sql(f"""
    SELECT COUNT(*)                                    AS rows_in_silver,
           COUNT(DISTINCT event_id)                    AS unique_key
    FROM {CATALOG}.{SILVER_SCHEMA}.sensor_data
"""))

## Data Quality Checks

In [0]:
# Sanity-check the ranges first
display(spark.sql(f"""
    SELECT MIN(pue) AS min_pue, MAX(pue) AS max_pue,
           MIN(consumption_kwh) AS min_kwh, MAX(consumption_kwh) AS max_kwh,
           MIN(reading_interval_s) AS min_int
    FROM {CATALOG}.{SILVER_SCHEMA}.sensor_data
"""))

In [0]:
# Domain rules Delta will enforce on every future write
spark.sql(f"""ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.sensor_data
             ADD CONSTRAINT pue_valid CHECK (pue >= 1.0 AND pue <= 3.0)""")

spark.sql(f"""ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.sensor_data
             ADD CONSTRAINT consumption_non_negative CHECK (consumption_kwh >= 0)""")

spark.sql(f"""ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.sensor_data
             ADD CONSTRAINT interval_positive CHECK (reading_interval_s > 0)""")

Can add a quarantine table to transfer bad rows

## Schema Evolution

In [0]:
# Controlled schema evolution: deliberately admitting ONE new column into silver.
# Explicit ALTER is auditable - unlike blind autoMerge, nothing sneaks in.
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.sensor_data
    ADD COLUMN avg_power_kw DECIMAL(10,2)
""")

In [0]:
# Reading the bronze source
bronze = spark.table(BRONZE_SENSOR)

# Deduplication rule: within each business key kepp the most recently ingested row
key_window = Window.partitionBy("event_id").orderBy(F.col("ingestion_ts").desc())

clean = (bronze
         # --- data quality filters that should not reach silver ---
         .filter(F.col("event_id").isNotNull()) # primary key must exist
         .filter(F.col("bidding_zone").isNotNull()) # FK must exist
         .filter(F.col("timestamp_utc").isNotNull()) # timestamp is required
         .withColumn("timestamp_utc", F.col("timestamp_utc").cast("timestamp")) # string > timestamp conversion
         # --- typing to match the silver schema established above ---
         .withColumn("consumption_kwh", F.col("consumption_kwh").cast("decimal(10,4)"))
         .withColumn("pue", F.col("pue").cast("decimal(4,3)"))
         .withColumn("reading_interval_s", F.col("reading_interval_s").cast("int"))
         .withColumn("avg_power_kw", F.col("avg_power_kw").cast("decimal(10,2)")) # Added new column
         # --- keep only the latest row per key ---
         .withColumn("row_num", F.row_number().over(key_window))
         .filter(F.col("row_num") == 1)
         # --- silver columns in the same order ---
         .selectExpr(
             "event_id",
             "timestamp_utc",
             "site_name",
             "country",
             "bidding_zone",
             "consumption_kwh",
             "pue",
             "avg_power_kw", # New column
             "enqueued_ts",
             "ingestion_ts",
             "site_id",
             "reading_interval_s",
             "current_timestamp() AS silver_processed_ts"        # when silver processed this row
    ))

clean.createOrReplaceTempView("silver_updates")

print("Rows after cleaning and deduplication:", clean.count())

In [0]:
# Re-run after adding an extra column

spark.sql(f"""
    MERGE INTO {CATALOG}.{SILVER_SCHEMA}.sensor_data AS t
    USING silver_updates AS s
      ON t.event_id = s.event_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# New column is visible
display(spark.sql(f"""
    SELECT *
    FROM {CATALOG}.{SILVER_SCHEMA}.sensor_data
    LIMIT 5
"""))